In [1]:
!pip install ldpc
!pip install bposd
!pip install mqt.qecc

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple


In [1]:
from ldpc.codes import rep_code
from bposd.hgp import hgp

construct surface code based on rep_code

In [2]:
h = rep_code(3)
surface_code = hgp(h1=h, h2=h, compute_distance=True)
surface_code.test()

print(surface_code.hz)
print(surface_code.N)
1, 0, 0, 1, 0, 0, 1,

<Unnamed CSS code>, (2,4)-[[13,1,3]]
 -Block dimensions: Pass
 -PCMs commute hz@hx.T==0: Pass
 -PCMs commute hx@hz.T==0: Pass
 -lx \in ker{hz} AND lz \in ker{hx}: Pass
 -lx and lz anticommute: Pass
 -<Unnamed CSS code> is a valid CSS code w/ params (2,4)-[[13,1,3]]
[[1 1 0 0 0 0 0 0 0 1 0 0 0]
 [0 1 1 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 1 1 0 0 0 0 1 0 1 0]
 [0 0 0 0 1 1 0 0 0 0 1 0 1]
 [0 0 0 0 0 0 1 1 0 0 0 1 0]
 [0 0 0 0 0 0 0 1 1 0 0 0 1]]
13


(1, 0, 0, 1, 0, 0, 1)

In [3]:
import numpy as np

# 高斯消元法（mod 2）
def gauss_elimination_mod2(A):
    n = len(A)
    m = len(A[0])
    Augmented = A.copy()
    col_trans = np.arange(m)
    for i in range(n):
        # 寻找主元
        if Augmented[i, i] == 0:
            # 如果主元为0，寻找下面一行有1的列交换
            # print(i,i)
            prior_jdx = 0
            min_nonzero_counts = n
            for j in range(i+1, m):
                if Augmented[i,j] == 1:
                    nonzero_counts = np.sum(Augmented[:,j])
                    if nonzero_counts < min_nonzero_counts:
                        prior_jdx = j
                        min_nonzero_counts = nonzero_counts
            j= prior_jdx
            col_trans[i],col_trans[j] = col_trans[j],col_trans[i]
            temp = Augmented[:,i].copy() 
            Augmented[:,i]  = Augmented[:,j]
            Augmented[:,j] = temp 
        # elif Augmented[i, i] == 1:
        #     # 如果主元为0，寻找下面一行有1的列交换
        #     # print(i,i)
        #     prior_jdx = i
        #     min_nonzero_counts = np.sum(Augmented[:,i])
        #     for j in range(i+1, m):
        #         if Augmented[i,j] == 1:
        #             nonzero_counts = np.sum(Augmented[:,j])
        #             if nonzero_counts < min_nonzero_counts:
        #                 prior_jdx = j
        #                 min_nonzero_counts = nonzero_counts
        #     j= prior_jdx
        #     if i == j:
        #         continue
        #     col_trans[i],col_trans[j] = col_trans[j],col_trans[i]
        #     temp = Augmented[:,i].copy() 
        #     Augmented[:,i]  = Augmented[:,j]
        #     Augmented[:,j] = temp 
    # print(Augmented)
    syndrome_transpose= np.identity(n,dtype=int)
    for i in range(n):
        # 对主元所在行进行消元
        if Augmented[i, i] == 1:
            for j in range(0, n):
                if j!=i and Augmented[j, i] == 1:
                    Augmented[j] ^= Augmented[i]
                    syndrome_transpose[j] ^= syndrome_transpose[i]

    return Augmented,col_trans,syndrome_transpose

def calculate_tran_syndrome(syndrome,syndrome_transpose):
    return syndrome_transpose @ syndrome %2

def calculate_original_error(our_result,col_trans):
    trans_results = np.zeros_like(our_result,dtype=int)
    col_trans = col_trans.tolist()
    for i in np.arange(len(col_trans)):
        trans_results[i] = our_result[col_trans.index(i)]
    return trans_results

In [4]:
surface_code.hz

array([[1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0],
       [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1],
       [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1]])

In [5]:
import numpy as np
hz_trans, col_trans, syndrome_transpose = gauss_elimination_mod2(surface_code.hz)

hz_trans


array([[1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0],
       [0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
       [0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0],
       [0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1],
       [0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1]])

In [6]:
syndrome_transpose

array([[1, 1, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0],
       [0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 1]])

In [7]:
col_trans

array([ 0,  1,  3,  5,  6,  8,  4,  7,  2,  9, 10, 11, 12])

In [8]:
B = hz_trans[:,len(hz_trans):len(hz_trans[0])]
B

array([[0, 0, 1, 1, 1, 0, 0],
       [0, 0, 1, 0, 1, 0, 0],
       [1, 0, 0, 1, 0, 1, 0],
       [1, 0, 0, 0, 1, 0, 1],
       [0, 1, 0, 0, 0, 1, 0],
       [0, 1, 0, 0, 0, 0, 1]])

initialize decoder
+ BP+OSD
+ UFDecoder

In [9]:
import numpy as np
from ldpc import bposd_decoder
from mqt.qecc import *  # UFDecoder

p = 0.1  # 错误率

# BP+OSD
bposd_decoder = bposd_decoder(
    surface_code.hz,
    error_rate=p,
    channel_probs=[None],
    max_iter=surface_code.N,
    bp_method="ms",
    ms_scaling_factor=0,
    osd_method="osd_cs",
    osd_order=7,
)

# UFDecoder
code = Code(surface_code.hx, surface_code.hz)
uf_decoder = UFHeuristic()
uf_decoder.set_code(code)

decode and calculate success rate
+ BP+OSD
+ UFDecoder (mqt-qecc)
+ Our method

In [10]:


H_X = surface_code.hz

weights = [
    np.log((1 - p) / p) for i in range(H_X.shape[1])
]  # 初始每个qubit的对数似然比

W_f = weights[: H_X.shape[0]]
W_g = weights[H_X.shape[0] :]

W_f_B = np.dot(W_f, B)  # W_f * B
W_g_B_W_f = W_f_B + W_g  # W_f * B + W_g
# print(f"W_g_B_W_f = {W_g_B_W_f}")

zero_g = np.where(
    W_g_B_W_f > 0,
    0,
    np.where(W_g_B_W_f < 0, 1, np.random.randint(0, 2, size=W_g_B_W_f.shape)),
)
# print(f"g = {g}")

B_g = np.dot(B, zero_g)  # B * g
print(f"B_g = {B_g}")

B_g = [0 0 0 0 0 0]


In [11]:
surface_code.lz

array([[1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0]])

In [13]:
num_trials = 10000

bposd_num_success = 0
uf_num_success = 0
our_num_success = 0

for i in range(num_trials):

    # generate error
    error = np.zeros(surface_code.N).astype(int)
    for q in range(surface_code.N):
        if np.random.rand() < p:
            error[q] = 1
    # num_errors = np.random.randint(1, surface_code.N // 2)  # 随机选择1到N//2个错误位置
    # error_indices = np.random.choice(surface_code.N, num_errors, replace=False)
    # error[error_indices] = 1  # 在随机选中的量子比特上设置错误
    # # print(f"Trial {i + 1}, error = {error}")

    # obtain syndrome
    syndrome = surface_code.hz @ error % 2
    # print(f"Trial {i + 1}, syndrome = {syndrome}")

    """Decode"""
    # 1. BP+OSD
    
    bposd_decoder.decode(syndrome)

    # 2. UFDecoder
    uf_decoder.decode(syndrome)
    uf_result = np.array(uf_decoder.result.estimate).astype(int)

    # 3. Our Decoder
    syndrome_copy = calculate_tran_syndrome(syndrome.copy(),syndrome_transpose)
    print(syndrome,syndrome_copy)
    bposd_result =  bposd_decoder.osdw_decoding
    g_index = col_trans[len(hz_trans):len(hz_trans[0])]
    bposd_g = [bposd_result[idx] for idx in g_index]
    True_g = [error[idx] for idx in g_index ]
    random_g = np.zeros_like(zero_g)
    for idx in range(len(random_g)):
        if np.random.rand() < p:
            random_g[idx] = 1
    f = (np.dot(B, zero_g) + syndrome_copy)%2
    our_result = np.hstack((f, zero_g))
    assert ((hz_trans @ our_result)%2 == syndrome_copy).all()
    # print(f"Trial {i + 1}, bposd result = {bposd_decoder.osdw_decoding}")
    # print(f"Trial {i + 1}, uf result = {uf_result}")
    
   
    # calculate success rate for each decoder
    bposd_residual_error = (bposd_decoder.osdw_decoding + error) % 2
    bpflag = (surface_code.lz @ bposd_residual_error % 2).any()
    if bpflag == 0:
        bposd_num_success += 1

    uf_residual_error = (uf_result + error) % 2
    flag = (surface_code.lz @ uf_residual_error % 2).any()
    if flag == 0:
        uf_num_success += 1
    
    trans_results = calculate_original_error(our_result,col_trans)
    assert ((surface_code.hz @ trans_results)%2 == syndrome).all(), (trans_results,error,bposd_result)
    # assert ((surface_code.hz @ bposd_result)%2 == syndrome).all()

    # print((surface_code.hz @ our_result)%2 ,syndrome)
    our_residual_error = (trans_results + error) % 2
    flag = (surface_code.lz @ our_residual_error % 2).any()
    if flag == 0:
        our_num_success += 1
    else:
        # print(surface_code.lz *trans_results,surface_code.lz *error)
        # print(trans_results,error)
        if bpflag==0:
            pass
            print(True_g,bposd_g)
            # print([bposd_result[1],bposd_result[4],bposd_result[7]]+list(bposd_result[9:13]))
        else:
            # print(True_g,bposd_g)
            pass
        pass

bposd_success_rate = bposd_num_success / num_trials
uf_success_rate = uf_num_success / num_trials
our_success_rate = our_num_success / num_trials
print(f"\nTotal trials: {num_trials}")
print(f"BP+OSD Success rate: {bposd_success_rate * 100:.2f}%")
print(f"UF Success rate: {uf_success_rate * 100:.2f}%")
print(f"Our Success rate: {our_success_rate * 100:.2f}%")

[0 0 1 1 0 0] [0 0 1 1 0 0]
[0 0 0 0 0 0] [0 0 0 0 0 0]
[1 1 0 1 1 0] [0 1 0 1 1 0]
[0 0 0 0 0 0] [0 0 0 0 0 0]
[1 1 1 0 0 1] [0 1 1 0 0 1]
[0 0 0 0 1 0] [0 0 0 0 1 0]
[0 0 0 1 0 1] [0 0 0 1 0 1]
[0 1 0 1 0 0] [1 1 0 1 0 0]
[1 0 0 1 0 1] [1 0 0 1 0 1]
[1 0 0 0 0 0] [1 0 0 0 0 0]
[1 1 0 0 0 0] [0 1 0 0 0 0]
[1 0 0 0 0 0] [1 0 0 0 0 0]
[0 1 0 0 0 0] [1 1 0 0 0 0]
[1 1 1 0 0 0] [0 1 1 0 0 0]
[0 0 0 0 0 0] [0 0 0 0 0 0]
[0 0 0 0 0 0] [0 0 0 0 0 0]
[0 0 1 0 0 0] [0 0 1 0 0 0]
[0 0 0 0 0 1] [0 0 0 0 0 1]
[0 0 1 0 0 1] [0 0 1 0 0 1]
[0 0 0 0 0 0] [0 0 0 0 0 0]
[0 0 0 0 0 0] [0 0 0 0 0 0]
[1 0 0 1 0 1] [1 0 0 1 0 1]
[0 0 0 0 1 1] [0 0 0 0 1 1]
[0 0 0 1 0 1] [0 0 0 1 0 1]
[0 0 0 0 1 0] [0 0 0 0 1 0]
[0 0 0 0 0 0] [0 0 0 0 0 0]
[0 0 0 0 0 0] [0 0 0 0 0 0]
[0 0 0 0 0 0] [0 0 0 0 0 0]
[0 0 0 0 1 0] [0 0 0 0 1 0]
[0 0 0 0 0 0] [0 0 0 0 0 0]
[0 0 0 0 0 0] [0 0 0 0 0 0]
[0 1 1 1 1 0] [1 1 1 1 1 0]
[0 0 0 0 1 1] [0 0 0 0 1 1]
[0 0 0 0 0 1] [0 0 0 0 0 1]
[0 0 1 1 1 1] [0 0 1 1 1 1]
[0 0 1 1 0 0] [0 0 1